# v4 — event decomposition

v3 values a swing with one regression: the run value of swinging at this pitch,
in this count. This notebook asks whether it is better to model what a swing
*produces* — a whiff, a foul, or a ball in play — and assemble the value from
those pieces:

```
Q_swing(s) = P(whiff | s)·RE(whiff, c) + P(foul | s)·RE(foul, c) + P(in play | s)·E[run value | in play, s]
```

The target is a function of (outcome, count), so both estimate the same
quantity. The premise is that whiffs are predictable (AUC 0.77 held out) while
the run value of a swing, pitch by pitch, barely beats a count-only lookup.

**The comparison contract.** Decomposition is credited only against a *direct
twin* that shares everything else: the score (`signed_edge`), pitch features,
hitter priors, folds, learner protocol, and the take model itself — the twins
call the same `fit_take`, and the notebook asserts their `Q_take` is identical,
so every difference comes from the swing branch.

**Learner protocol.** Every sub-model in both twins is fitted the same way:
XGBoost, depth 7, learning rate 0.05, rounds set by early stopping on 10% of
training games held back. v3 used v1's fixed settings (learning rate 0.01, 200
rounds); section 1 measures what that change alone does.

**Win conditions, fixed before the numbers.** Against its twin, decomposition
wins if it (a) improves construct validity, (b) improves reliability without a
construct-validity loss, (c) lowers contamination, (d) enables a
personalization route the direct model cannot express, or (e) predicts
held-out conditional means better. A tie means the direct model stays.

Held out throughout, on the personalized folds (train 2022 → 2023, 2022–23 →
2024, 2022–24 → 2025). 2026 is not loaded.

In [1]:
import sys; sys.path.insert(0, '..')
import numpy as np, pandas as pd

from src import data as D, evaluate as E, features as F, decision as DEC
from src.baselines import fit_v1, fit_direct, predict_both, V1_FEATURES
from src.decomposition import fit_decomposed

pd.set_option('display.width', 180); pd.set_option('display.max_columns', 30)
SEASONS = range(2021, 2026)          # 2026 is kept out of model selection; not loaded here
PF = E.PERSONALIZED_FOLDS
SHOW = E.SHOW
V3_FEATURES = ['plate_x_bat','plate_z_norm','count','stand','p_throws',
               'release_speed','pfx_x','pfx_z','pitch_type']
PRIORS = ['prior_whiff', 'prior_foul']

In [2]:
df = D.drop_pitchers_batting(D.load_seasons(SEASONS, verbose=False))
df = D.add_zone_frame(df.dropna(subset=['plate_x_mid','plate_z_mid']))
for c in ('stand','p_throws','pitch_type'):
    df[c] = df[c].astype('category')

# Hitter priors, each season from the two seasons before it (2021 is prior data only).
key = ['game_pk', 'at_bat_number', 'pitch_number']
pdf = F.season_hot_zone(df).merge(F.season_contact_priors(df)[key + PRIORS],
                                  on=key, how='left', validate='one_to_one')
assert pdf[PRIORS].notna().all().all()
print(f'{len(pdf):,} pitches, seasons {sorted(pdf.season.unique())}')

2,827,989 pitches, seasons [np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]


In [3]:
def row(run, label, score=DEC.DEFAULT_SCORE):
    """One scorecard row: the player-metric checks plus both swing calibration slopes."""
    s = E.harness(run, label, score)['summary']
    s['swing bin slope'] = s.pop('swing calib slope')
    s['swing pred slope'] = round(E.prediction_calibration(run.held, 'swing', ci=False).slope.mean(), 3)
    return s

SUB = ['swing signal recovered', 'swing bin slope', 'swing pred slope']

def rounds(run):
    return pd.DataFrame({season: m.rounds for season, m in run.models.items()}).T

## 1. The learner change on its own

Before any decomposition, the direct model is refit under early stopping, with
nothing else changed — v3's features, with and without the hot zone, v3's
routing. This is a hyperparameter-only comparison.

In [4]:
learner = {}
for feats_label, feats in [('full pitch frame', V3_FEATURES), ('+ hot zone', V3_FEATURES + ['hot_zone'])]:
    learner[(feats_label, 'v1 settings')] = E.run_folds(pdf, feats, PF)
    learner[(feats_label, 'early stopping')] = E.run_folds(pdf, feats, PF, fit=fit_direct)
t1 = pd.DataFrame([row(run, f'{a} | {b}') for (a, b), run in learner.items()]).set_index('variant')
t1[SUB + SHOW]

,swing signal recovered,swing bin slope,swing pred slope,chase | zone_swing,zone_swing | chase,split-half r,YoY R2 (mean),Zone% |r|,next-season r (partial)
variant,,,,,,,,,
full pitch frame | v1 settings,0.886,1.271,1.264,-0.923,0.723,0.817,0.590,0.274,0.107
full pitch frame | early stopping,0.948,1.071,1.053,-0.916,0.711,0.815,0.584,0.269,0.104
+ hot zone | v1 settings,0.892,1.291,1.235,-0.895,0.677,0.838,0.624,0.184,0.154
+ hot zone | early stopping,0.953,1.078,1.025,-0.885,0.662,0.844,0.629,0.169,0.167


In [5]:
# Rounds chosen by early stopping, per fold.
rounds(learner[('+ hot zone', 'early stopping')])

,take,swing
2023,125,79
2024,155,91
2025,250,121


In [6]:
# Swing calibration with 95% intervals (games resampled), full pitch frame + hot zone.
cal = {}
for setting in ['v1 settings', 'early stopping']:
    h = learner[('+ hot zone', setting)].held
    cal[(setting, 'by location x count')] = E.bin_calibration(h, 'swing', ci=True)[['slope', 'slope_lo', 'slope_hi']]
    cal[(setting, 'by predicted value')] = E.prediction_calibration(h, 'swing')[['slope', 'slope_lo', 'slope_hi', 'pred_range', 'obs_range']]
pd.concat(cal, names=['learner', 'grouping']).round(3)

slope  slope_lo  slope_hi  pred_range  obs_range
learner        grouping            season                                                  
v1 settings    by location x count 2023    1.398     1.332     1.454         NaN        NaN
                                   2024    1.251     1.199     1.317         NaN        NaN
                                   2025    1.223     1.169     1.272         NaN        NaN
               by predicted value  2023    1.259     1.230     1.284       0.160      0.190
                                   2024    1.212     1.187     1.241       0.160      0.197
                                   2025    1.234     1.207     1.260       0.159      0.197
early stopping by location x count 2023    1.190     1.134     1.235         NaN        NaN
                                   2024    1.036     0.994     1.090         NaN        NaN
                                   2025    1.007     0.961     1.045         NaN        NaN
               by predicted value  2023    1.054     1.026     1.077       0.194      0.194
                                   2024    1.009     0.988     1.031       0.196      0.201
                                   2025    1.012     0.990     1.034       0.200      0.202

**The swing model's miscalibration was undertraining.** Under v1's settings the
swing model's predictions are too compressed: grouped by predicted value, the
observed means spread about a quarter more than the predictions (slope
1.21–1.26, every interval well above 1). Early stopping picks 79–121 rounds at
a learning rate five times higher, and the slope falls to 1.01–1.05 — within
its interval of 1 in 2024 and 2025. Grouped by location and count the same
happens (1.22–1.40 → 1.01–1.19), and the share of location structure recovered
rises from 0.89 to 0.95. The one season still above 1 is 2023, whose fold
trains on 2022 alone.

**It barely moves the metric without the hot zone, and helps a little with
it.** For the full pitch frame every player-metric check moves by 0.012 or
less. With the hot zone, next-season validity rises 0.154 → 0.167 and Zone%
contamination falls 0.184 → 0.169, for a small construct cost (−0.895 / +0.677
→ −0.885 / +0.662).

The direct model under early stopping is the twin for everything below.

## 2. B2 — generic twins

Full pitch frame, no hitter features in either twin. This isolates
decomposition itself.

In [7]:
direct = learner[('full pitch frame', 'early stopping')]
decomp = E.run_folds(pdf, V3_FEATURES, PF, fit=fit_decomposed)
assert np.array_equal(direct.held.q_take.to_numpy(), decomp.held.q_take.to_numpy()), 'twins must share Q_take'
print('Q_take identical across twins: yes')
t2 = pd.DataFrame([row(direct, 'direct'), row(decomp, 'decomposed')]).set_index('variant')
t2[SUB + SHOW]

Q_take identical across twins: yes


,swing signal recovered,swing bin slope,swing pred slope,chase | zone_swing,zone_swing | chase,split-half r,YoY R2 (mean),Zone% |r|,next-season r (partial)
variant,,,,,,,,,
direct,0.948,1.071,1.053,-0.916,0.711,0.815,0.584,0.269,0.104
decomposed,0.949,1.093,1.044,-0.914,0.706,0.816,0.583,0.270,0.100


In [8]:
rounds(decomp)

,take,outcome,bip_value
2023,125,305,70
2024,155,535,69
2025,250,578,66


In [9]:
# What the decomposed classifier gets right: whiff / foul / in play, held out.
E.outcome_diagnostics(decomp.held).round(4)

,swings,logloss,logloss_base_rate,whiff_auc,improvement_%,whiff_slope,foul_slope,bip_slope
season,,,,,,,,
2023,339205,0.9659,1.0851,0.7640,10.9846,1.0232,0.9591,1.0156
2024,337782,0.9617,1.0833,0.7675,11.2261,1.0078,0.9673,1.0052
2025,336884,0.9584,1.0831,0.7697,11.5097,1.0244,0.9927,1.0211


In [10]:
# Both twins on the conditional means, with intervals.
pd.concat({(twin, grouping): f(run.held, 'swing', **kw)[['slope', 'slope_lo', 'slope_hi']]
           for twin, run in [('direct', direct), ('decomposed', decomp)]
           for grouping, f, kw in [('by location x count', E.bin_calibration, {'ci': True}),
                                   ('by predicted value', E.prediction_calibration, {})]},
          names=['twin', 'grouping']).round(3)

slope  slope_lo  slope_hi
twin       grouping            season                           
direct     by location x count 2023    1.165     1.108     1.212
                               2024    1.035     0.991     1.087
                               2025    1.012     0.967     1.052
           by predicted value  2023    1.111     1.086     1.134
                               2024    1.019     0.997     1.041
                               2025    1.030     1.008     1.054
decomposed by location x count 2023    1.167     1.111     1.211
                               2024    1.065     1.020     1.120
                               2025    1.046     0.999     1.091
           by predicted value  2023    1.087     1.063     1.112
                               2024    1.008     0.986     1.031
                               2025    1.038     1.015     1.061

In [11]:
# How much do the two twins disagree, pitch by pitch?
a, b = direct.held, decomp.held
print(f"corr(Q_swing): {np.corrcoef(a.q_swing, b.q_swing)[0, 1]:.4f}   "
      f"corr(edge): {np.corrcoef(a.edge, b.edge)[0, 1]:.4f}")
print(f"recommended action differs on {(np.sign(a.edge) != np.sign(b.edge)).mean():.2%} of pitches")
sa, sb = E.player_metric(a, 'signed_edge'), E.player_metric(b, 'signed_edge')
m = sa.merge(sb, on=['season', 'batter'])
print(f"hitter-season decision values correlate {np.corrcoef(m.decision_value_x, m.decision_value_y)[0, 1]:.4f}")

corr(Q_swing): 0.9435   corr(edge): 0.9918
recommended action differs on 2.27% of pitches
hitter-season decision values correlate 0.9980


### Δ stability

Five refits of each twin on fold 3, each on a bootstrap sample of training
games, scored on the same 2025 pitches. A pitch is unstable if the recommended
action changes sign across refits. Same procedure for both twins.

In [12]:
def delta_stability(fit, features, frame, refits=5, seed=1126):
    train_seasons, valid = PF[-1]
    need = [f for f in features if f != 'count']
    rv = D.run_value_table(frame, train_seasons)
    part = D.apply_run_value(frame[frame.season.isin([*train_seasons, valid])].dropna(subset=need),
                             rv, name='target').dropna(subset=['target'])
    train, held = part[part.season.isin(train_seasons)], part[part.season == valid]
    by_game = train.groupby('game_pk').indices
    games = np.array(list(by_game))
    rng = np.random.default_rng(seed)
    edges = []
    for _ in range(refits):
        idx = np.concatenate([by_game[g] for g in rng.choice(games, len(games), replace=True)])
        model = fit(train.iloc[idx], features=features, rv=rv)
        edges.append(predict_both(held, model).edge.to_numpy())
    E_ = np.vstack(edges)
    unstable = (np.sign(E_) != np.sign(E_[0])).any(0)
    close = np.abs(E_.mean(0)) < 0.02
    return {'unstable, all pitches': unstable.mean(), 'unstable, |edge| < 0.02': unstable[close].mean(),
            'share of pitches with |edge| < 0.02': close.mean(), 'edge SD across refits': E_.std(0).mean()}

stab = pd.DataFrame({'direct': delta_stability(fit_direct, V3_FEATURES, pdf),
                     'decomposed': delta_stability(fit_decomposed, V3_FEATURES, pdf)})
stab.round(4)

,direct,decomposed
"unstable, all pitches",0.0647,0.0573
"unstable, |edge| < 0.02",0.4628,0.4554
share of pitches with |edge| < 0.02,0.1140,0.1144
edge SD across refits,0.0086,0.0070


**B2 is a tie.** On every player-metric check the twins differ by 0.005 or
less, and the decomposed twin's calibration intervals overlap the direct
twin's. The two recommend a different action on 2.3% of pitches, and their
hitter-season decision values correlate 0.998: they are, to within noise, the
same metric.

The decomposed classifier itself is good and well calibrated — 11.0–11.5%
better log loss than the base rates, whiff AUC 0.764–0.770, per-class slopes
0.96–1.02 — but that accuracy was already available to the direct regression
implicitly.

**The one difference is stability.** Refit on resampled games, the decomposed
twin's Δ varies less (SD 0.0070 against 0.0086 runs), and fewer pitches change
recommended action across refits (5.7% against 6.5%). Almost all of the
instability, in both twins, sits on the 11% of pitches whose Δ is within 0.02
runs of zero, where the choice barely matters. Stability is not among the win
conditions fixed in advance, so it does not change the verdict: against (a)–(e)
the decomposition does not win, and the direct model stays for the generic
case.

## 3. B4 — personalized twins

Two hitter priors, each built from the two seasons before the one scored, with
the same smoothing and shrinkage as the hot zone:

- **damage** — `hot_zone`, the hitter's expected exit velocity by location
  (from v3);
- **contact** — `prior_whiff` and `prior_foul`, the hitter's whiff and foul
  rates per swing by location. New here.

In the decomposed twin each goes only to the sub-model it describes: contact
priors to the whiff / foul / in-play classifier, the hot zone to the in-play
value model. The direct twin gets both in its one swing regression. Neither
twin's take model sees them.

First, whether the contact priors are worth building at all: does a hitter's
surface reproduce the following season?

In [13]:
sw = F.swings_with_outcome(df)          # every season, 2021 included, as history
counts = sw.groupby(['season', 'batter']).size()
rows = []
for b in [2023, 2024, 2025]:
    keep = set(counts[b][counts[b] >= 300].index) & set(counts[b - 1][counts[b - 1] >= 300].index)
    target = F.surface_on_grid(F.value_surface(sw[(sw.season == b) & sw.batter.isin(keep)], ['_whiff', '_foul']))
    for label, window in [('one-season prior', [b - 1]), ('two-season prior', [b - 2, b - 1])]:
        prior = F.surface_on_grid(F.value_surface(sw[sw.season.isin(window) & sw.batter.isin(keep)], ['_whiff', '_foul']))
        common = sorted(set(target) & set(prior))
        T, P = np.stack([target[h] for h in common]), np.stack([prior[h] for h in common])
        for k, name in enumerate(['whiff', 'foul']):
            r = np.nanmean([np.corrcoef(T[:, n, k], P[:, n, k])[0, 1] for n in range(T.shape[1])])
            rows.append({'target season': b, 'prior': label, 'rate': name, 'hitters': len(common), 'r': r})
pd.DataFrame(rows).pivot_table(index=['rate', 'target season'], columns='prior', values='r').round(3)

prior                one-season prior  two-season prior
rate  target season                                    
foul  2023                      0.415             0.432
      2024                      0.422             0.457
      2025                      0.437             0.464
whiff 2023                      0.642             0.657
      2024                      0.646             0.669
      2025                      0.627             0.660

**Both contact priors reproduce, and a two-season window beats one.** A
hitter's whiff-rate surface predicts the next season's at r ≈ 0.66–0.67 (0.63–0.65
from one season) — about as stable as the hot zone. Foul rate is less stable,
r ≈ 0.43–0.46. Both are kept for the comparison below.

In [14]:
B4 = {
    'direct + hot zone':                    (fit_direct, V3_FEATURES + ['hot_zone']),
    'decomposed + hot zone':                (fit_decomposed, V3_FEATURES + ['hot_zone']),
    'direct + hot zone + contact priors':     (fit_direct, V3_FEATURES + ['hot_zone'] + PRIORS),
    'decomposed + hot zone + contact priors': (fit_decomposed, V3_FEATURES + ['hot_zone'] + PRIORS),
}
b4 = {label: (learner[('+ hot zone', 'early stopping')] if label == 'direct + hot zone'
              else E.run_folds(pdf, feats, PF, fit=fit))
      for label, (fit, feats) in B4.items()}
for a_, b_ in [('direct + hot zone', 'decomposed + hot zone'),
               ('direct + hot zone + contact priors', 'decomposed + hot zone + contact priors')]:
    assert np.array_equal(b4[a_].held.q_take.to_numpy(), b4[b_].held.q_take.to_numpy()), 'twins must share Q_take'
t3 = pd.DataFrame([row(run, label) for label, run in b4.items()]).set_index('variant')
t3[SUB + SHOW]

,swing signal recovered,swing bin slope,swing pred slope,chase | zone_swing,zone_swing | chase,split-half r,YoY R2 (mean),Zone% |r|,next-season r (partial)
variant,,,,,,,,,
direct + hot zone,0.953,1.078,1.025,-0.885,0.662,0.844,0.629,0.169,0.167
decomposed + hot zone,0.961,1.077,0.988,-0.877,0.648,0.845,0.632,0.162,0.171
direct + hot zone + contact priors,0.953,1.078,1.022,-0.878,0.641,0.847,0.624,0.174,0.164
decomposed + hot zone + contact priors,0.965,1.059,0.960,-0.862,0.582,0.847,0.615,0.184,0.173


In [15]:
pd.concat({label: E.outcome_diagnostics(b4[label].held) for label in
           ['decomposed + hot zone', 'decomposed + hot zone + contact priors']},
          names=['variant']).round(4)

swings  logloss  logloss_base_rate  whiff_auc  improvement_%  whiff_slope  foul_slope  bip_slope
variant                                season                                                                                                  
decomposed + hot zone                  2023    339205   0.9659             1.0851     0.7640        10.9846       1.0232      0.9591     1.0156
                                       2024    337782   0.9617             1.0833     0.7675        11.2261       1.0078      0.9673     1.0052
                                       2025    336884   0.9584             1.0831     0.7697        11.5097       1.0244      0.9927     1.0211
decomposed + hot zone + contact priors 2023    339205   0.9550             1.0851     0.7767        11.9921       0.9987      0.9323     0.9857
                                       2024    337782   0.9493             1.0833     0.7811        12.3670       0.9930      0.9554     0.9943
                                       2025    336884   0.9457             1.0831     0.7825        12.6811       1.0090      0.9750     1.0078

In [16]:
stab4 = pd.DataFrame({'direct + hot zone': delta_stability(fit_direct, V3_FEATURES + ['hot_zone'], pdf),
                      'decomposed + hot zone': delta_stability(fit_decomposed, V3_FEATURES + ['hot_zone'], pdf)})
stab4.round(4)

,direct + hot zone,decomposed + hot zone
"unstable, all pitches",0.0649,0.0575
"unstable, |edge| < 0.02",0.4666,0.4452
share of pitches with |edge| < 0.02,0.1156,0.1185
edge SD across refits,0.0087,0.0073


**Decomposition still ties once personalized.** With the hot zone, the
decomposed twin is marginally more reliable (split-half 0.845 against 0.844,
YoY 0.632 against 0.629) and less contaminated (Zone% 0.162 against 0.169),
with next-season validity 0.171 against 0.167 — but it gives up construct
validity (−0.877 / +0.648 against −0.885 / +0.662). Condition (b) requires no
construct loss, and the other gaps are all 0.007 or less, so this is a tie
too. Stability again favours the decomposition
(unstable pitches 5.8% against 6.5%).

**Contact priors hurt the metric while improving the model.** They make the
classifier better — whiff AUC 0.764–0.770 → 0.777–0.782, log loss 11.0–11.5% →
12.0–12.7% better than base rate — but in the decomposed twin they pull the
score from the decision toward the hitter: zone-swing | chase falls 0.648 →
0.582, YoY 0.632 → 0.615, Zone% rises 0.162 → 0.184. In the direct twin they
do little but still cost construct validity (+0.662 → +0.641). A hitter who
rarely whiffs gets a higher `Q_swing` everywhere, so his swings score better
regardless of whether they were good decisions. This is the risk the plan
flagged, and it materialised. **Contact priors are dropped.** Condition (d)
— a personalization route only the decomposition can express — is met in
form, but the route it enables makes the metric worse.

## 4. Against the benchmarks

Everything held out, 2023–2025. The two plate-discipline rates are computed
from the same pitches: O-Swing% (lower is better, so negated) and Z-Swing% −
O-Swing%. Their construct-validity cells are circular — they are built from
chase and zone-swing rates — so only reliability, contamination and prediction
are comparable.

In [17]:
v1_run = E.run_folds(pdf, V1_FEATURES, PF)
bar = learner[('+ hot zone', 'v1 settings')]

h = v1_run.held
inz = D.in_rulebook_zone(h, ball_edge=True)
k_ = ['season', 'batter']
rates = pd.DataFrame({'o_swing': h[~inz].groupby(k_, observed=True).swing.mean(),
                      'z_swing': h[inz].groupby(k_, observed=True).swing.mean(),
                      'pitches': h.groupby(k_, observed=True).size()}).reset_index()
rates = rates[rates.pitches >= E.QUALIFY_PITCHES]

def as_scores(values):
    s = rates[k_ + ['pitches']].assign(raw=values)
    z = s.groupby('season').raw.transform(lambda v: (v - v.mean()) / v.std(ddof=0))
    return s.assign(decision_value=100 + 10 * z)

def checks_from_scores(scores, label):
    return {'variant': label,
            'YoY R2 (mean)': round(E.yoy_reliability(scores).r2.mean(), 3),
            'Zone% |r|': round(E.zone_pct_correlation(h, scores).r.abs().mean(), 3),
            'next-season r (partial)': round(E.predictive_validity(h, scores).r_partial.mean(), 3)}

bench = pd.DataFrame([
    {**E.metric_checks(v1_run.held, 'y_pred')['summary'], 'variant': 'v1 as designed (chosen_value)'},
    row(bar, 'Track A bar (v1 settings, + hot zone)'),
    row(b4['direct + hot zone'], 'direct + hot zone (early stopping)'),
    row(b4['decomposed + hot zone'], 'decomposed + hot zone'),
    checks_from_scores(as_scores(-rates.o_swing), 'O-Swing% (negated)'),
    checks_from_scores(as_scores(rates.z_swing - rates.o_swing), 'Z-Swing% − O-Swing%'),
]).set_index('variant')
bench[SHOW]

,chase | zone_swing,zone_swing | chase,split-half r,YoY R2 (mean),Zone% |r|,next-season r (partial)
variant,,,,,,
v1 as designed (chosen_value),-0.883,0.669,0.757,0.526,0.081,0.137
"Track A bar (v1 settings, + hot zone)",-0.895,0.677,0.838,0.624,0.184,0.154
direct + hot zone (early stopping),-0.885,0.662,0.844,0.629,0.169,0.167
decomposed + hot zone,-0.877,0.648,0.845,0.632,0.162,0.171
O-Swing% (negated),NaN,NaN,NaN,0.697,0.421,0.077
Z-Swing% − O-Swing%,NaN,NaN,NaN,0.557,0.114,0.146


## Findings

All figures held out, 2023–2025, on the personalized folds.

**Decomposition ties the direct model, generic and personalized.** The two
twins produce effectively the same metric — hitter-season decision values
correlate 0.998 — and no win condition fixed in advance is met. By the rule set
before the numbers, **the direct model stays.** The decomposed classifier is
accurate and calibrated, but a direct regression with the same features and
enough rounds already captures what it knows. Its one advantage is lower
estimation variance (Δ SD across refits 0.0070 against 0.0086), which was not
a win condition.

**The real finding is the learner.** v1's fixed settings left the swing model
too compressed — the "understated contrast" seen in v3. Early stopping
calibrates it (slope by predicted value 1.21–1.26 → 1.01–1.05), and with the
hot zone it raises next-season validity to 0.167 and lowers Zone%
contamination to 0.169, against 0.154 and 0.184 for the Track A bar, at a small
construct cost.

**Contact priors reproduce but corrupt the score.** Whiff- and foul-rate
surfaces are stable traits and improve outcome prediction, but they move the
score toward "is he a good contact hitter": construct validity and YoY fall and
contamination rises. They are dropped.

**Against simple benchmarks.** The model beats O-Swing% on every comparable
check — O-Swing% has Zone% |r| 0.421, meaning it largely measures how many
strikes a hitter is thrown. Z-Swing% − O-Swing% is a stronger baseline than
expected: less contaminated (0.114 against 0.169) and nearly as predictive
(0.146 against 0.167), though less reliable (YoY 0.557 against 0.629). It is
the benchmark to beat on contamination.

**The current model** is therefore the direct twin with the hot zone under
early stopping: full pitch frame, hot zone to the swing model only,
`signed_edge`.

| | chase\|zs | zs\|chase | split-half | YoY R² | Zone% \|r\| | next-season |
|---|---|---|---|---|---|---|
| Track A bar (v1 settings) | −0.895 | 0.677 | 0.838 | 0.624 | 0.184 | 0.154 |
| **current: direct + hot zone, early stopping** | −0.885 | 0.662 | **0.844** | **0.629** | **0.169** | **0.167** |
| decomposed + hot zone | −0.877 | 0.648 | 0.845 | 0.632 | 0.162 | 0.171 |